# 1.3 Your turn: your own chat

Continues from [`01.2-irc-chat.ipynb`](01.2-irc-chat.ipynb) — same pipeline shape, pointed at
your own export instead of the IRC showcase.

The one extra step is anonymisation. `humanize` maps each real name to a stable nickname, so
you can hand in your work without handing over your friends' names.

In [ ]:
from IPython.display import display

from goad_toolkit.datatransforms import Pipeline, RegexFeature, TimeFeatures
from notebooktester import param

from wa_analyzer.data import PROCESSED, load_own_chat

`RegexFeature` is imported here, not rewritten. The class you derived in `01.2-irc-chat.ipynb`
is the one `goad_toolkit` ships — same three modes, same `feature`-not-`name` parameter, same
coverage log on `extract`. Writing it once was the exercise; retyping it in every notebook
that needs it is just a copy waiting to drift out of step with the original.

## Where `own` comes from

`load_own_chat()` reads `config.toml`'s `current` key — the parquet file that this notebook
writes at the bottom of a run. The normal loop is: run the preprocessor, run this notebook
once, then set `current` in `config.toml` to the filename it just wrote, so every later
notebook loads it automatically.

If you have more than one export and want to switch between them without touching the config,
pass a filename directly instead: `load_own_chat(filename="whatsapp-20250101-120000.parq")`
loads exactly that file from `data/processed/` and ignores `current` entirely.

If this is your first run and neither exists yet, see the README's **Run the preprocessor**
section before continuing — the next cell errors with that same pointer if there is nothing
to load.

In [ ]:
own = load_own_chat(filename="whatsapp-20260909-185004.csv")
own.head()

In [ ]:
anon = {
    "Jeffrey Halters": "Student",
    "Rick": "Partner",
}
own["author"] = own["author"].replace(anon)
print(f"{len(anon)} authors, anonymised. For example:")
display(own.head(3))

Now the *same pipeline shape* as `01.2-irc-chat.ipynb`, pointed at a different dataframe.
That is the payoff of having written the steps as objects rather than as cells: nothing
changes but the input.

In [ ]:
own_pipeline = Pipeline()
own_pipeline.add(TimeFeatures, column="timestamp")
own_pipeline.add(RegexFeature, name="urls",
                 column="message", pattern=r"https?://\S+",
                 feature="has_url", mode="has")

own_enriched = own_pipeline.apply(own)
display(own_enriched.head())

> **Your turn.** Add at least two more steps to `own_pipeline` above — they do not have to
> be `RegexFeature`. A count (`str.count`), a group membership, a summary statistic, another
> `TimeFeatures`-style calendar column: anything shaped like a `TransformBase` step works.
> Pick things specific to *your* chat: an emoji, a group in-joke, a language you switch into,
> the way one person always signs off.
>
> Chat data is unusually rich for this: a WhatsApp export gives you timestamps with daily and
> weekly rhythms, natural language, and half a dozen distributions with textbook shapes. Most
> of what makes this course work comes from features you extract yourself in this notebook.

In [ ]:
import pandas as pd
from goad_toolkit.datatransforms import TransformBase

# Klasse voor het Dagritme (ochtend, middag, avond, nacht)
class DayPart(TransformBase):
    """Categoriseren van tijd naar dagdelen (ieder 6 uur)."""
    def transform(
        self, data: pd.DataFrame, column: str = "timestamp", feature: str = "day_part",
    ) -> pd.DataFrame:
        hours = pd.to_datetime(data[column]).dt.hour #Haalt uit de tijdstempel puur het uur van de dag (een getal van 0 t/m 23).
        data[feature] = pd.cut( #Knipt deze uren op in bakjes (bins). Omdat pandas telt als (van, tot en met]
            hours,
            bins=[-1, 5, 11, 17, 23],
            labels=["Nacht", "Ochtend", "Middag", "Avond"],
        )
        return data

# Klasse voor Weekdag vs. Weekend
class PeriodFeature(TransformBase):
  """Categoriseer berichten naar Weekdag of Weekend."""

  def transform(
      self, data: pd.DataFrame, column: str = "timestamp"
  ) -> pd.DataFrame:
    ts = pd.to_datetime(data[column])
    # 0=maandag t/m 6=zondag; 5 en 6 zijn weekend
    data["is_weekend"] = ts.dt.weekday.isin([5, 6])
    data["period"] = data["is_weekend"].map(
        {True: "Weekend (za-zo)", False: "Weekdag (ma-vr)"}
    )
    return data

# Klasse voor pure emoji-/smiley-berichten (zonder tekst)
class EmojiOnly(TransformBase):
    """Markeer berichten die alleen uit emojis of leestekens bestaan (geen letters/cijfers)."""
    def transform(
        self, data: pd.DataFrame, column: str = "message", feature: str = "only_emoji",
    ) -> pd.DataFrame:
        data[feature] = ~data[column].str.contains(r"[a-zA-Z0-9]", regex=True) #zoekt naar kleine letters, grote letters en cijfers. Alles wat daar buitenvalt is false (leestekens en emoji's)
        return data

# Klasse voor reactietijd met sprekerswisselingen, dag-reset en 8 uurgrens
class ResponseTimeFeature(TransformBase):
  """Berekent reactietijd in minuten met sprekerswisseling, dag-reset en 8-uursgrens."""
  def transform(
      self,
      data: pd.DataFrame,
      author_col: str = "author",
      ts_col: str = "timestamp",
      feature: str = "response_time_min",
  ) -> pd.DataFrame:
    # 1. Zorg voor datetime en datum
    ts = pd.to_datetime(data[ts_col])
    date = ts.dt.date
    # 2. Kijk naar het vorige bericht
    prev_author = data[author_col].shift(1)
    prev_ts = ts.shift(1)
    prev_date = date.shift(1)
    # 3. Bereken het verschil in minuten
    delta_min = (ts - prev_ts).dt.total_seconds() / 60.0
    # 4. De 3 voorwaarden van Jeffrey:
    # - Andere auteur (geen reactie op jezelf)
    # - Zelfde dag (reset 's nachts)
    # - Maximaal 8 uur (480 minuten)
    is_valid_reply = (
        (data[author_col] != prev_author)
        & (date == prev_date)
        & (delta_min > 0)
        & (delta_min <= 480)
    )
    # 5. Vul de kolom: alleen minuten als het een geldige reactie is, anders NaN
    data[feature] = delta_min.where(is_valid_reply, other=float("nan"))
    return data
    
# Alle stappen toevoegen aan de pipeline + lege berichten verwijderen + labelen berichten met media + berichten met alleen maar media
own_pipeline.add(DayPart)
own_pipeline.add(PeriodFeature)
own_pipeline.add(EmojiOnly)
own_pipeline.add(ResponseTimeFeature)
own_pipeline.add(
    RegexFeature,
    name="deleted",
    column="message",
    pattern=r"verwijderd",
    feature="is_deleted",
    mode="has", #true / false 
)
own_pipeline.add(
    RegexFeature,
    name="media",
    column="message",
    pattern=r"bijgevoegd",
    feature="has_media",
    mode="has", #true / false 
)
own_pipeline.add(
    RegexFeature,
    name="only_media",
    column="message",
    pattern=r":\s*[^a-zA-Z0-9\s]*<bijgevoegd",
    feature="has_only_media",
    mode="has", #true / false 
)
your_turn = own_pipeline.apply(own) # 1. pas alle stappen toe op je data
new_columns = set(your_turn.columns) - set(own_enriched.columns) # 2. Kijk welke kolommen er nieuw bij zijn gekomen

MIN_NEW_FEATURES = param(2, test=0) # Controles Raoul
assert len(new_columns) >= MIN_NEW_FEATURES, (  # noqa: S101 -- the check *is* the point here
    f"add at least {MIN_NEW_FEATURES} more own_pipeline.add(...) steps above, so this produces "
    f"columns beyond {sorted(own_enriched.columns)}"
)
print(f"new column(s): {sorted(new_columns) or 'none yet -- add steps above'}") # Toon resultaten
display(your_turn.head())

In [ ]:
import re
from goad_toolkit.datatransforms import TransformBase
import numpy as np
import pandas as pd


class ReplyWordCountAggregator(TransformBase):
    """Berekent schone woordentelling per bericht (zonder leestekens/emoji's),
    koppelt de berichtlengte van de zender aan de reactietijd van de ontvanger,
    en extraheert alle geldige directe 1-op-1 reactie-interacties.
    """

    def __init__(self, max_wait_hours: float = 8.0):
        self.max_wait_hours = max_wait_hours

    @staticmethod
    def count_clean_words(text: str) -> int:
        """Telt uitsluitend inhoudelijke alfanumerieke woorden (geen emoji's of leestekens)."""
        if not isinstance(text, str):
            return 0
        # Match woorden met letters, cijfers en diakrieten
        words = re.findall(r"\b[a-zA-Z0-9_À-ÿ]+\b", text)
        return len(words)

    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        ts = pd.to_datetime(df["timestamp"])
        date = ts.dt.date

        # 1. Schone woordentelling van het huidige bericht
        df["word_count"] = df["message"].apply(self.count_clean_words)

        # 2. Koppel kenmerken van het voorgaande bericht
        prev_author = df["author"].shift(1)
        prev_words = df["word_count"].shift(1)
        prev_date = date.shift(1)
        prev_ts = ts.shift(1)
        delta_min = (ts - prev_ts).dt.total_seconds() / 60.0

        # 3. Filtercriteria voor een zuivere directe reactie:
        # - Wisseling van auteur (geen monoloog / opeenvolgende berichten van dezelfde auteur)
        # - Zelfde kalenderdag (geen vertekende nachtpauzes)
        # - Maximaal 8 uur wachttijd (480 minuten)
        # - Voorgaand bericht bevat minimaal 1 inhoudelijk woord
        max_min = self.max_wait_hours * 60.0
        is_valid_reply = (
            (df["author"] != prev_author)
            & (date == prev_date)
            & (delta_min > 0)
            & (delta_min <= max_min)
            & (prev_words > 0)
            & df["message"].notna()
        )

        # 4. Stel de reactie-paren dataset samen
        replies = pd.DataFrame({
            "timestamp": ts[is_valid_reply],
            "date": date[is_valid_reply],
            "author": df["author"][is_valid_reply],       # Ontvanger (degene die reageert)
            "sender": prev_author[is_valid_reply],        # Zender van de stimulus
            "prev_words": prev_words[is_valid_reply].astype(int), # X-as: woordlengte van de zender
            "response_time_min": delta_min[is_valid_reply],       # Y-as: reactietijd in minuten
            "log10_time": np.log10(delta_min[is_valid_reply].clip(lower=0.05)), # Log10-reactietijd
            "my_words": df["word_count"][is_valid_reply].astype(int),
        }).reset_index(drop=True)

        return replies


# 4. Voer de extractie uit op de zojuist verrijkte 'your_turn'
aggregator = ReplyWordCountAggregator(max_wait_hours=8.0)
replies_words = aggregator.transform(your_turn)

# 5. Veilig opslaan naar een apart bestand in data/processed/
# LET OP: Dit overschrijft 'chat-20260915-190537.parq' NIET en raakt 02.5, 03.3 en 04.4 niet!
replies_outfile = PROCESSED / "chat-replies-words.parq"
replies_words.to_parquet(replies_outfile, index=False)

print(f"Succesvol {len(replies_words):,} directe reactie-paren weggeschreven naar {replies_outfile}")
print(f"- Jeffrey reageert op Rick: {(replies_words['author'] == 'Student').sum():,} reacties")
print(f"- Rick reageert op Jeffrey: {(replies_words['author'] == 'Partner').sum():,} reacties")
display(replies_words.head(10))


In [ ]:
from datetime import datetime

outfile = PROCESSED / f"chat-{datetime.now():%Y%m%d-%H%M%S}.parq"
your_turn.to_parquet(outfile, index=False)
print(f"Wrote {outfile}")
print("Put that filename after `current` in config.toml so the other notebooks find it.")

## 1.4 What to write down

Before moving on, you should be able to answer these about your own data. Not in your head —
written down, because lesson 2 starts by assuming them.

1. **What is one row, now?** One message. Say what a message is in your export — does a
   photo count, a system notice, a message that was deleted?
2. **How many rows did you lose, and to what?** Every parse drops something. Name the number
   and the reason.
3. **How many *people* are in your data?** Not messages — people. That number is much
   smaller, and in lesson 2 you will find out how much it matters.
4. **Which features did you add, and what question is each for?** A feature you cannot
   attach to a question is one you will never use.
5. **What is in there that should not be?** Bots, group-admin notices, one person's phone
   posting twice. You do not have to remove them yet. You have to know they are there.

---

**Where this goes next.** The features you built here are the input to everything that
follows: lesson 2 compares them across people, lesson 3 across time, lesson 4 asks what shape
they have, and lesson 5 asks which of them tell people apart. And the habit — enrich before
you model — is the one the machine learning course builds on directly.